In [1]:
# necessary imports 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import logging
import sys

# plotting imports
import pandapower as pp
import pandapower.topology as top
import pandapower.plotting as plot
import simbench as sb

# simbench imports
import simbench as sb

# DWD imports (German weather data), open source
import urllib.request
import zipfile, io

In [ ]:
# we know already from the paper that the project SimBench was active between 2015 and 2019; 
# in order to obtain actual temp. data, we now need to confirm what year the data corresponds to

In [2]:
# find out from what years the simbench data originates 
path_lp = '/opt/anaconda3/envs/p2p/lib/python3.10/site-packages/simbench/networks/1-complete_data-mixed-all-0-sw/LoadProfile.csv'
lp_raw = pd.read_csv(path_lp, sep=';', nrows=5)
print("First rows of time column:")
print(lp_raw.iloc[:5, 0])  # first column is time

# Also check the full index range
lp_full = pd.read_csv(path_lp, sep=';', index_col=0)
print(f"\nIndex type: {type(lp_full.index)}")
print(f"First value: {lp_full.index[0]}")
print(f"Last value:  {lp_full.index[-1]}")
print(f"Total rows:  {len(lp_full)}")
print(f"35136 / 4 / 24 = {35136/4/24:.0f} days = 1 year")

First rows of time column:
0    01.01.2016 00:00
1    01.01.2016 00:15
2    01.01.2016 00:30
3    01.01.2016 00:45
4    01.01.2016 01:00
Name: time, dtype: object

Index type: <class 'pandas.core.indexes.base.Index'>
First value: 01.01.2016 00:00
Last value:  31.12.2016 23:45
Total rows:  35136
35136 / 4 / 24 = 366 days = 1 year


In [ ]:
# this shows that the data we have available in Simbench is form the year 2016; it corresponds exactly to one year

In [7]:
def dwd_hourly_temp(station_id, year):
    import urllib.request, zipfile, io, re
    base = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/hourly/air_temperature/historical/"
    
    with urllib.request.urlopen(base) as r:
        html = r.read().decode()
    
    fname = re.findall(rf'stundenwerte_TU_{station_id:05d}_\d+_\d+_hist\.zip', html)
    if not fname:
        raise ValueError(f"Station {station_id} not found in DWD index")
    
    url = base + fname[0]
    print(f"Downloading: {fname[0]}")
    
    with urllib.request.urlopen(url) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
        data_file = [f for f in z.namelist() if f.startswith('produkt_')][0]
        df = pd.read_csv(z.open(data_file), sep=';')
    
    # DWD stores datetime as integer YYYYMMDDHH — parse explicitly
    df['datetime'] = pd.to_datetime(df['MESS_DATUM'].astype(str).str.strip(),
                                    format='%Y%m%d%H')
    df['temp_C'] = pd.to_numeric(df['TT_TU'].astype(str).str.strip(),
                                  errors='coerce')
    
    df = df[df['datetime'].dt.year == year][['datetime','temp_C']]
    df = df.dropna().reset_index(drop=True)
    
    print(f"  Got {len(df)} hourly records for {year}")
    print(f"  Range: {df['datetime'].iloc[0]} → {df['datetime'].iloc[-1]}")
    print(f"  Temp: {df['temp_C'].min():.1f}°C to {df['temp_C'].max():.1f}°C")
    
    return df

temp_hannover = dwd_hourly_temp(2564, 2016)
temp_luebeck  = dwd_hourly_temp(2928, 2016)

Downloading: stundenwerte_TU_02564_20020101_20251231_hist.zip
  Got 8784 hourly records for 2016
  Range: 2016-01-01 00:00:00 → 2016-12-31 23:00:00
  Temp: -8.3°C to 30.2°C
Downloading: stundenwerte_TU_02928_19510101_20251231_hist.zip
  Got 8784 hourly records for 2016
  Range: 2016-01-01 00:00:00 → 2016-12-31 23:00:00
  Temp: -11.2°C to 34.9°C


In [8]:
# Step 2 — Upsample to 15-min to match SimBench
def to_15min(df):
    df = df.set_index('datetime')
    df.index = pd.to_datetime(df.index)
    return df.resample('15min').interpolate('linear')

temp_han_15 = to_15min(temp_hannover)
temp_lub_15 = to_15min(temp_luebeck)
print(f"After upsampling: {len(temp_han_15)} steps (expect 35136)")

After upsampling: 35133 steps (expect 35136)
